# 02 · Train (or load) the model + full analysis

Loads the dataset built by `01_build_dataset.ipynb` (the committed `data/example/` by
default, or your own data in the same schema), then runs the original pipeline end-to-end:
tokenize → train-or-load → held-out evaluation → confidence calibration → inference →
clinical-style summary.

- **Apply our published model (default):** keep `train_from_scratch = False` in section 6.
  It loads [`SamAgnoli/deberta-v3-base-spatial-language-detection`](https://huggingface.co/SamAgnoli/deberta-v3-base-spatial-language-detection).
- **Fine-tune your own:** set `train_from_scratch = True` (needs a GPU runtime).

> Before running, click Runtime -> Change Runtime type -> change to T4 GPU.


## 1. Setup

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn accelerate sentencepiece protobuf

In [ ]:
import os, re, random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
import evaluate

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# === Setup: locate the dataset + matcher (Colab auto-clones; local uses the repo) ===
import os, json, re, string

def _find_data_root():
    for d in ("../data", "data", "spatial-language-classifier/data",
              "/content/spatial-language-classifier/data"):
        if os.path.isdir(d):
            return os.path.abspath(d)
    return None

DATA_ROOT = _find_data_root()
if DATA_ROOT is None:                       # fresh Colab: fetch the repo
    os.system("git clone -q https://github.com/SamAgnoli/spatial-language-classifier.git")
    DATA_ROOT = _find_data_root()
assert DATA_ROOT, "Couldn't find the repo's data/ folder \u2014 is the repo public on GitHub?"

# Folder with train.csv / val.csv / test.csv. Default = the committed example; to use your
# own "step 1 ready" data, upload it and set DATA_DIR to its folder.
DATA_DIR     = os.environ.get("SPATIAL_DATA_DIR", os.path.join(DATA_ROOT, "example"))
MATCHER_PATH = os.environ.get("SPATIAL_MATCHER",  os.path.join(DATA_ROOT, "spatial_matcher.json"))
# Model to load when train_from_scratch = False (section 6).
MODEL_SOURCE = os.environ.get("SPATIAL_MODEL", "SamAgnoli/deberta-v3-base-spatial-language-detection")
# Local working dir for training artifacts (only used when train_from_scratch = True).
proj_dir = os.environ.get("SPATIAL_PROJ_DIR", ".")
os.makedirs(os.path.join(proj_dir, "models"), exist_ok=True)

print("DATA_DIR     ->", DATA_DIR)
print("MATCHER_PATH ->", MATCHER_PATH)
print("MODEL_SOURCE ->", MODEL_SOURCE)


In [ ]:
# --- Load the step-1 dataset into the variables the original pipeline expects ---
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
for _d in (df_train, df_val, df_test):
    _d['session_id'] = _d['session_id'].astype(str)
print('Loaded splits:', len(df_train), 'train /', len(df_val), 'val /', len(df_test), 'test')

# --- Rebuild the dictionary matcher from the serialized artifact ---
# EXACT / PREFIXES come from 01_build_dataset; the helper bodies below are copied VERBATIM
# from section 2b so candidate detection is bit-for-bit identical.
with open(MATCHER_PATH, 'r', encoding='utf-8') as _f:
    _m = json.load(_f)
EXACT = set(_m['EXACT'])
PREFIXES = list(_m['PREFIX'])

_STRIP = string.punctuation + ' \t\n'

def is_candidate(word: str) -> bool:
    w = word.lower().strip(_STRIP)
    if not w:
        return False
    if w in EXACT:
        return True
    for stem in PREFIXES:
        if w.startswith(stem):
            return True
    return False

WORD_RE_DICT = re.compile(r"[A-Za-z']+")

def count_candidates(utt):
    return sum(1 for w in WORD_RE_DICT.findall(str(utt)) if is_candidate(w))

print('Matcher: %d EXACT, %d PREFIX entries' % (len(EXACT), len(PREFIXES)))

# --- Auto-fill `is_candidate` ONLY if your data is missing it -------------------------
# NOTE: this does nothing for the shipped example (which already has the column). It runs
# *only* when `is_candidate` is absent -- i.e., you are fine-tuning on your OWN new data
# that did NOT pass through step 1 (01_build_dataset). In that case it tags each `coded_word`
# with the dictionary matcher above, so the candidates-only evaluation views still work.
# If the column already exists, your data is left untouched.
for _name, _d in [('train', df_train), ('val', df_val), ('test', df_test)]:
    if 'is_candidate' not in _d.columns:
        _d['is_candidate'] = _d['coded_word'].apply(is_candidate)
        print(f'  is_candidate not found in {_name}: auto-filled {len(_d)} rows (custom training data).')


## 4. Build the HuggingFace `DatasetDict` and tokenize

Input format: `(utterance, coded_word)` pair → RoBERTa encodes as `<s> utterance </s></s> coded_word </s>`.

In [ ]:
model_name = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

keep_cols = ['utterance', 'coded_word', 'spatial_or_not', 'session_id', 'line', 'is_candidate']

def to_hf(d):
    return Dataset.from_pandas(d[keep_cols].copy(), preserve_index=False)

spatial_dataset = DatasetDict({
    'train': to_hf(df_train),
    'validation': to_hf(df_val),
    'test': to_hf(df_test),
})

spatial_dataset = spatial_dataset.rename_column('spatial_or_not', 'labels')

def tokenize_batch(batch):
    return tokenizer(
        batch['utterance'],
        batch['coded_word'],
        truncation=True,
        max_length=128,
    )

tokenized = spatial_dataset.map(tokenize_batch, batched=True)
meta_cols = ['session_id', 'line', 'coded_word', 'utterance', 'is_candidate']
tokenized_for_model = tokenized.remove_columns(meta_cols)
print(tokenized_for_model)

## 5. Metrics

In [ ]:
acc_m = evaluate.load('accuracy')
prec_m = evaluate.load('precision')
rec_m = evaluate.load('recall')
f1_m = evaluate.load('f1')

def evaluate_binary(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':  acc_m.compute(predictions=preds, references=labels)['accuracy'],
        'precision': prec_m.compute(predictions=preds, references=labels, pos_label=1, zero_division=0)['precision'],
        'recall':    rec_m.compute(predictions=preds, references=labels, pos_label=1, zero_division=0)['recall'],
        'f1':        f1_m.compute(predictions=preds, references=labels, pos_label=1)['f1'],
        'macro_f1':  f1_m.compute(predictions=preds, references=labels, average='macro')['f1'],
    }

Set `train_from_scratch = False` (the default) to **load the published model** from the Hub (`MODEL_SOURCE`) and run the full analysis without retraining. Set it to `True` to **fine-tune DeBERTa-v3 fresh** on your `train.csv` (needs a GPU runtime); the new model is then saved under `{proj_dir}/models/spatial_deberta`.

In [ ]:
train_from_scratch = False #True if retraining
saved_model_dir = f'{proj_dir}/models/spatial_deberta'

training_args = TrainingArguments(
    output_dir=f'{proj_dir}/models',
    eval_strategy='epoch',
    save_strategy='epoch',
    num_train_epochs=2,  # was 5; 2 epochs is enough to prevent overfitting/maintain DeBERTa's reasoning abilities
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    fp16=True,  # ~2x faster + memory headroom on the free T4
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    seed=SEED,
    logging_steps=50,
)

if train_from_scratch:
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, id2label={0: "not_spatial", 1: "spatial"}, label2id={"not_spatial": 0, "spatial": 1}, torch_dtype=torch.float32)  # force fp32 weights so fp16=True AMP works (deberta-v3 ships fp16)
else:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_SOURCE, num_labels=2, torch_dtype=torch.float32)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_for_model['train'],
    eval_dataset=tokenized_for_model['validation'],
    compute_metrics=evaluate_binary,
    data_collator=DataCollatorWithPadding(tokenizer, return_tensors='pt'),
)

if train_from_scratch:
    trainer.train()
    trainer.save_model(saved_model_dir)
    tokenizer.save_pretrained(saved_model_dir)
    print('Saved to', saved_model_dir)

## 7. Held-out test evaluation

Touched exactly once: this is the honest generalisation number. The saved CSV lets you spot-check failure cases.

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=tokenized_for_model['test'])
print('Held-out test metrics:')
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

In [ ]:
pred_out = trainer.predict(tokenized_for_model['test'])
logits = pred_out.predictions
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
preds = probs.argmax(axis=-1)
labels = pred_out.label_ids

# Pull metadata back from the (pre-removal) tokenized test set so we can join is_candidate.
is_cand = np.array(tokenized['test']['is_candidate'])

test_meta = pd.DataFrame({
    'session_id': tokenized['test']['session_id'],
    'line':       tokenized['test']['line'],
    'coded_word': tokenized['test']['coded_word'],
    'utterance':  tokenized['test']['utterance'],
    'is_candidate': is_cand,
    'true_label': labels,
    'pred_label': preds,
    'pred_prob_spatial': probs[:, 1],
})

# Stash the raw test logits so the calibration section can reuse them without
# re-running trainer.predict on the test split.
test_logits_raw = logits

def report(name, mask):
    if mask.sum() == 0:
        print(f'\n=== {name}: no rows ===')
        return
    print(f'\n=== {name}  (n={int(mask.sum())}) ===')
    print(classification_report(
        labels[mask], preds[mask],
        labels=[0, 1],
        target_names=['not_spatial', 'spatial'],
        digits=3, zero_division=0,
    ))
    cm = confusion_matrix(labels[mask], preds[mask], labels=[0, 1])
    print(pd.DataFrame(cm,
                       index=['true_not', 'true_spatial'],
                       columns=['pred_not', 'pred_spatial']))

# Headline: dictionary candidates only — this is the meaningful view.
report('Dictionary candidates only (headline)', is_cand)
# Secondary: overall (every word, including trivially non-candidate ones)
report('Overall (every word in held-out utterances)', np.ones_like(is_cand, dtype=bool))

# CSV save is deferred to the calibration section below so the on-disk file
# carries both raw and calibrated columns in a single artifact.

## 7.5 Per-word confidence calibration

The trained classifier above gives the right *decisions* but its raw probabilities are **over-confident** — a raw 0.98 on a fine-tuned RoBERTa typically only corresponds to ~80–90% empirical accuracy. This section adds an interpretable per-word confidence so that, for any word, the number can be read literally ("≈80% likely spatial") and you can spot the cases where the model is genuinely unsure.

**Method:** temperature scaling — fit one scalar `T` on the validation split that minimises NLL, then divide every logit by `T` before the sigmoid. It is the standard post-hoc calibrator for modern neural networks (Guo, Pleiss, Sun & Weinberger, ICML 2017, [arXiv:1706.04599](https://arxiv.org/abs/1706.04599)).

**Why this is safe:** temperature scaling is **monotonic**. It never reorders words or flips an argmax decision — only the magnitudes of the probabilities change. The trained model and the held-out test metrics above stay bit-for-bit identical; the section ends with an assertion that proves it.

**What gets added to `test_predictions.csv`:**

- `confidence_spatial` — calibrated P(spatial). 0.05 = highly confident *not* spatial; 0.95 = highly confident spatial.
- `confidence_decision` — `max(p, 1-p)`. Confidence in whichever way the model decided. 0.95 either direction = confident; 0.55 = a near coin-flip the model isn't sure about. **Sort the CSV by this column to find the model's least-certain predictions.**
- `calib_method`, `temperature_T` — bookkeeping (always `"temp"` and the fitted `T`).

In [ ]:
# 7a. Pull validation + test logits.
# Validation logits are needed because we fit the calibrator on them (the model
# never trained on validation, and the test split stays untouched). Test logits
# are reused from the cell above — no need to re-run trainer.predict on test.
from scipy.optimize import minimize_scalar
from sklearn.metrics import brier_score_loss

val_out    = trainer.predict(tokenized_for_model['validation'])
val_logits = val_out.predictions
val_z      = val_logits[:, 1] - val_logits[:, 0]      # binary decision logit
val_y      = val_out.label_ids.astype(int)
val_p_raw  = 1.0 / (1.0 + np.exp(-val_z))
val_cand   = np.asarray(tokenized['validation']['is_candidate'], dtype=bool)

test_z     = test_logits_raw[:, 1] - test_logits_raw[:, 0]
test_y     = labels.astype(int)
test_p_raw = 1.0 / (1.0 + np.exp(-test_z))
test_cand  = np.asarray(tokenized['test']['is_candidate'], dtype=bool)

assert len(test_z) == len(test_meta), 'row mismatch between test logits and test_meta'

print(f'val:  {len(val_z):5d} words ({int(val_cand.sum())} dictionary candidates)')
print(f'test: {len(test_z):5d} words ({int(test_cand.sum())} dictionary candidates)')

In [ ]:
# 7b. Expected Calibration Error helper.
# Bins predictions, takes |mean_confidence − observed_accuracy| in each bin,
# averages weighted by bin size. Reports both uniform and quantile binning
# because uniform bins go nearly empty in the middle for a confident classifier.
def expected_calibration_error(y_true, y_prob, n_bins=10, strategy='uniform'):
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)
    if strategy == 'uniform':
        edges = np.linspace(0.0, 1.0, n_bins + 1)
    elif strategy == 'quantile':
        edges = np.unique(np.quantile(y_prob, np.linspace(0.0, 1.0, n_bins + 1)))
        if len(edges) < 2:
            edges = np.array([0.0, 1.0])
    else:
        raise ValueError("strategy must be 'uniform' or 'quantile'")
    nb  = len(edges) - 1
    idx = np.clip(np.digitize(y_prob, edges, right=True), 1, nb) - 1
    N, ece, rows = len(y_prob), 0.0, []
    for b in range(nb):
        mask = idx == b
        n = int(mask.sum())
        if n == 0:
            continue
        conf = y_prob[mask].mean()
        acc  = y_true[mask].mean()
        gap  = abs(conf - acc)
        ece += (n / N) * gap
        rows.append({'bin': f'[{edges[b]:.2f},{edges[b+1]:.2f}]', 'n': n,
                     'confidence': round(conf, 4), 'accuracy': round(acc, 4),
                     'gap': round(gap, 4)})
    return ece, pd.DataFrame(rows)

In [ ]:
# 7c. Diagnose the raw probabilities. ECE = expected calibration error (lower
# is better; 0 = perfect). Brier blends calibration + discrimination.
# The candidates-only row is the one to trust — the overall ECE is deceptively
# tiny because easy non-spatial words pile up near 0.
all_mask = np.ones(len(test_y), dtype=bool)

def diagnose(y, p, mask, label):
    e_u, _ = expected_calibration_error(y[mask], p[mask], n_bins=10, strategy='uniform')
    e_q, _ = expected_calibration_error(y[mask], p[mask], n_bins=10, strategy='quantile')
    b      = brier_score_loss(y[mask], p[mask])
    print(f'{label:30s} n={int(mask.sum()):5d}  '
          f'ECE(uniform)={e_u:.4f}  ECE(quantile)={e_q:.4f}  Brier={b:.4f}')

print('RAW probabilities on the held-out test set:')
diagnose(test_y, test_p_raw, all_mask,  '  overall (every word)')
diagnose(test_y, test_p_raw, test_cand, '  candidates only')

print('\nPer-bin reliability (candidates, uniform 10 bins):')
_, raw_tab = expected_calibration_error(test_y[test_cand], test_p_raw[test_cand], n_bins=10)
print(raw_tab.to_string(index=False))

In [ ]:
# 7d. Fit temperature scaling on the validation split — CANDIDATES ONLY.
# T = argmin NLL(sigmoid(val_z / T), val_y). One scalar, monotonic transform.
#
# We fit on val_z[val_cand] rather than all val_z because ~87% of validation
# words are trivial non-candidates auto-labeled 0 (the dictionary already rules
# them out). Including them biases T toward whatever calibrates the easy mass,
# which dilutes calibration on the candidate words where the model is doing real
# disambiguation work. Fitting on candidates only typically yields a larger T and
# a lower candidate-ECE; overall ECE may tick up slightly (acceptable trade).
def _nll_temp(T, z, y):
    p = 1.0 / (1.0 + np.exp(-z / T))
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

res = minimize_scalar(_nll_temp, args=(val_z[val_cand], val_y[val_cand]),
                      bounds=(0.05, 10.0), method='bounded')
T = float(res.x)
print(f'Fitted temperature T = {T:.4f}   (fit on {int(val_cand.sum())} candidate validation words)')
if T > 1.05:
    print('  T > 1 → the raw model was OVER-confident on candidates; calibration pulls probabilities toward 0.5.')
elif T < 0.95:
    print('  T < 1 → the raw model was UNDER-confident on candidates (unusual for fine-tuned DeBERTa — worth investigating).')
else:
    print('  T ≈ 1 → raw probabilities were already reasonably calibrated on candidates; calibration has little effect.')

In [ ]:
# 7e. Apply the fitted temperature to the test split and report before/after.
# The candidate-only row is the meaningful one (overall ECE is dominated by easy negatives).
test_p_temp = 1.0 / (1.0 + np.exp(-test_z / T))

def compare(mask, label):
    print(f'--- {label} (n={int(mask.sum())}) ---')
    print(f"  {'method':6s} {'ECE_uniform':>12s} {'ECE_quantile':>13s} {'Brier':>8s}")
    for name, p in [('raw', test_p_raw), ('temp', test_p_temp)]:
        e_u, _ = expected_calibration_error(test_y[mask], p[mask], n_bins=10, strategy='uniform')
        e_q, _ = expected_calibration_error(test_y[mask], p[mask], n_bins=10, strategy='quantile')
        b      = brier_score_loss(test_y[mask], p[mask])
        print(f'  {name:6s} {e_u:12.4f} {e_q:13.4f} {b:8.4f}')

compare(test_cand, 'candidates only (judge by this)')
print()
compare(all_mask,  'overall (every word)')

# Decision-preservation check: temperature scaling is monotonic, so the argmax
# decision for every word must be identical to the model's raw prediction.
# This is the formal guarantee that calibration did not change model behavior.
temp_preds = (test_p_temp >= 0.5).astype(int)
assert (temp_preds == test_meta['pred_label'].values).all(), \
    'Calibration changed at least one decision — this should be impossible for monotonic temperature scaling.'
print('\nDecision-preservation check passed: every test prediction is identical before and after calibration.')

In [ ]:
# 7f. Attach calibrated columns to test_meta and save the consolidated CSV.
# This is the artifact the project consumes downstream.
test_meta['calib_method']        = 'temp'
test_meta['temperature_T']       = T
test_meta['confidence_spatial']  = test_p_temp                              # calibrated P(spatial)
test_meta['confidence_decision'] = np.maximum(test_p_temp, 1.0 - test_p_temp)  # confidence in whichever way the model went

pred_csv = f'{proj_dir}/test_predictions.csv'
test_meta.to_csv(pred_csv, index=False)
print(f'Saved {len(test_meta)} rows (raw + calibrated columns) to:\n  {pred_csv}')

cols = ['coded_word', 'true_label', 'pred_label', 'confidence_spatial',
        'confidence_decision', 'utterance']
print('\nMost confident SPATIAL judgments among candidates:')
display(test_meta[test_meta['is_candidate']]
        .sort_values('confidence_spatial', ascending=False)[cols].head(10))

print('\nLeast-confident judgments — the cases where the model is genuinely unsure:')
display(test_meta[test_meta['is_candidate']]
        .sort_values('confidence_decision')[cols].head(10))

In [ ]:
# 7g. Reliability diagram — raw vs calibrated, on the candidate subset.
# If calibration worked, the calibrated points sit closer to the diagonal.
import matplotlib.pyplot as plt

_, raw_tab = expected_calibration_error(test_y[test_cand], test_p_raw[test_cand],  n_bins=10)
_, cal_tab = expected_calibration_error(test_y[test_cand], test_p_temp[test_cand], n_bins=10)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], '--', color='gray', label='perfect calibration')
ax.plot(raw_tab['confidence'], raw_tab['accuracy'], 'o-', label='raw')
ax.plot(cal_tab['confidence'], cal_tab['accuracy'], 's-', label=f'calibrated (temp, T={T:.2f})')
ax.set_xlabel('mean predicted confidence')
ax.set_ylabel('observed fraction spatial')
ax.set_title('Reliability — dictionary candidates')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 8. Inference: dictionary-gated per-word labels with calibrated confidence

`classify_utterance(utterance)` tokenizes the utterance, **filters to dictionary candidates only** (using `is_candidate` from cell 2b), then labels each candidate as spatial or not in context.

Non-dictionary words (e.g. "what", "did", "we") are omitted from the output — the model is only ever asked about words it was trained on.

Each result also carries a **calibrated confidence** (`confidence_spatial`, `confidence_decision`), using the temperature `T` fitted in section 7.5. `print_predictions` flags any word whose `confidence_decision < 0.65` with `← unsure`, so you can immediately see which calls the model isn't sure about.

For the example `"what did we find when we went in there?"`, both `in` and `there` are dictionary candidates and should come back as `spatial_or_not = 1`.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device).eval()

WORD_RE = re.compile(r"[A-Za-z']+")  # words only; skips punctuation and digits

def classify_utterance(utterance: str, batch_size: int = 32):
    """Return per-word spatial labels for dictionary-candidate words only.

    Non-candidate words (e.g. 'what', 'did', 'we') are omitted from the output.
    Each result carries both the raw model probability and the *calibrated*
    confidence (from temperature T fitted in section 7.5) so unsure predictions
    are visible at inference time. The hard 0/1 decision is unchanged — it's
    the model's raw argmax — so live behavior matches the test metrics.
    """
    tokens = WORD_RE.findall(utterance)
    cands = [(i, w) for i, w in enumerate(tokens) if is_candidate(w)]
    if not cands:
        return []
    positions = [i for i, _ in cands]
    words = [w for _, w in cands]
    results = []
    with torch.no_grad():
        for start in range(0, len(words), batch_size):
            end = start + batch_size
            chunk_w = words[start:end]
            chunk_p = positions[start:end]
            enc = tokenizer(
                [utterance] * len(chunk_w),
                chunk_w,
                truncation=True,
                max_length=128,
                padding=True,
                return_tensors='pt',
            ).to(device)
            logits = model(**enc).logits                                # raw logits
            raw_probs = torch.softmax(logits, dim=-1).cpu().numpy()
            z = (logits[:, 1] - logits[:, 0]).cpu().numpy()             # binary decision logit
            cal_p = 1.0 / (1.0 + np.exp(-z / T))                        # calibrated P(spatial)
            for pos, w, raw_row, p_cal in zip(chunk_p, chunk_w, raw_probs, cal_p):
                results.append({
                    'word': w,
                    'position': pos,
                    'spatial_or_not': int(raw_row.argmax()),
                    'prob_spatial_raw':    float(raw_row[1]),
                    'confidence_spatial':  float(p_cal),
                    'confidence_decision': float(max(p_cal, 1.0 - p_cal)),
                })
    return results

def print_predictions(utterance: str):
    print(f'Utterance: {utterance}')
    preds = classify_utterance(utterance)
    if not preds:
        print('  (no dictionary-candidate words)')
        return
    for r in preds:
        flag = '[SPATIAL]' if r['spatial_or_not'] == 1 else '[       ]'
        # confidence_decision near 0.5 = unsure; near 1.0 = confident either way
        unsure = ' ← unsure' if r['confidence_decision'] < 0.65 else ''
        print(f"  {flag} pos={r['position']:2d}  {r['word']:15s}  "
              f"p_spatial={r['confidence_spatial']:.3f}  "
              f"confidence={r['confidence_decision']:.3f}{unsure}")

# --- Worked example from the brief ---
print_predictions('what did we find when we went in there?')
print()

# --- A few additional cases ---
print_predictions('Put the red wheel on top of the block and then push it across the table.')
print()
print_predictions("I'll see you in 2018.")

In [ ]:
# Sanity-check inference against the held-out test set: pick a few utterances
# the model has NEVER seen and compare predicted spatial words vs. human codes.
sample_lines = (
    test_meta.groupby(['session_id', 'line'])
             .first()
             .reset_index()
             .sample(3, random_state=SEED)
)

for _, row in sample_lines.iterrows():
    utt = row['utterance']
    truth = test_meta[(test_meta['session_id'] == row['session_id']) &
                      (test_meta['line'] == row['line'])][['coded_word', 'true_label']]
    print('=' * 80)
    print('Utterance:', utt)
    print('Human codes:')
    print(truth.to_string(index=False))
    print('Model predictions (per word):')
    preds_r = classify_utterance(utt)
    spatial_words = [r['word'] for r in preds_r if r['spatial_or_not'] == 1]
    print('  spatial words →', spatial_words)

## 9. Error analysis (optional)

Top most-confidently-wrong predictions on the held-out test set. Useful for spotting systematic mistakes (e.g., the model over-predicting spatial for 'in' regardless of context).

In [ ]:
errors = test_meta[test_meta['true_label'] != test_meta['pred_label']].copy()
errors['confidence'] = errors.apply(
    lambda r: r['pred_prob_spatial'] if r['pred_label'] == 1 else 1 - r['pred_prob_spatial'],
    axis=1,
)
errors = errors.sort_values('confidence', ascending=False).head(20)
errors[['coded_word', 'true_label', 'pred_label', 'pred_prob_spatial', 'utterance']]

## 10. Summary metrics

Condenses the held-out test results into one diagnostic-accuracy table:
sensitivity (recall/TPR), specificity (TNR), PPV (precision), NPV, accuracy,
F1, and Cohen's kappa, derived from the 2x2 confusion matrix. Reported for two
views: dictionary **candidates only** (the headline view) and **overall**
(every word). Uses the raw argmax predictions (`labels`, `preds`, `is_cand`)
from the test-evaluation cell above. Every rate is zero-division safe.

In [ ]:
# 10. Summary metrics (sensitivity / specificity / PPV / NPV / kappa).
# Built from the same raw test predictions used in the headline eval above.
# Flip a view's mask to all-True for the overall picture; candidates-only is the
# meaningful view since non-candidate words are trivially not_spatial.
from sklearn.metrics import confusion_matrix, cohen_kappa_score, f1_score


def clinical_summary(y_true, y_pred):
    """2x2 confusion matrix -> diagnostic-accuracy metrics (zero-division safe)."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    n = tn + fp + fn + tp
    div = lambda num, den: num / den if den else float('nan')
    return {
        'n':           n,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'accuracy':    div(tp + tn, n),
        'sensitivity': div(tp, tp + fn),          # recall / TPR
        'specificity': div(tn, tn + fp),          # TNR
        'ppv':         div(tp, tp + fp),          # precision
        'npv':         div(tn, tn + fn),
        'f1':          f1_score(y_true, y_pred, labels=[0, 1], zero_division=0),
        'kappa':       cohen_kappa_score(y_true, y_pred, labels=[0, 1]),
    }


views = {
    'candidates_only': is_cand,
    'overall':         np.ones_like(is_cand, dtype=bool),
}

summary = pd.DataFrame(
    {name: clinical_summary(labels[mask], preds[mask]) for name, mask in views.items()}
).T

count_cols = ['n', 'tp', 'fp', 'fn', 'tn']
rate_cols  = ['accuracy', 'sensitivity', 'specificity', 'ppv', 'npv', 'f1', 'kappa']
summary[count_cols] = summary[count_cols].astype(int)

print('=== Counts ===')
print(summary[count_cols].to_string())
print()
print('=== Diagnostic-accuracy metrics ===')
print(summary[rate_cols].round(3).to_string())